In [6]:
import matplotlib.pyplot as plt
import requests
import xml.etree.ElementTree as ET
import geopandas as gpd
import pandas as pd
import numpy as np

In [3]:
# Upload shp file
gdf = gpd.read_file("/Users/nihankaragul/Documents/Documents - Nihan’s MacBook Air/2024 Fall/Dissertation/Data/raw_shp/tur_polbnda_adm3_southeast/tur_polbnda_adm3_Southeast.shp")
gdf = gdf[~gdf['adm1_en'].isin(['MUS', 'BATMAN', 'ELAZIG'])]

# Convert it to whatever epsg they are using
gdf = gdf.to_crs(epsg=3857)

In [29]:
collapse = pd.read_excel("/Users/nihankaragul/Documents/Documents - Nihan’s MacBook Air/2024 Fall/Dissertation/Data/Processed/collapsed_orig.xlsx")


collapse

,Layer,Feature_ID,qc_id,fid,tip,adm1_en,adm1_tr,adm1,adm2_tr,adm2_en,...,excluded from assessment,no damage,not assessed,collapsed,out of scope,unidentified (unable to enter),registered/protected,moderate damage,total_dmg,total.bld
0,Number of (heavily and more) damaged buildings...,0,0,3,Kırsal Mahalle,HATAY,DORTYOL,TUR031,DÖRTYOL,DORTYOL,...,10,215,0,0,0,0,0,0,6,5
1,Number of (heavily and more) damaged buildings...,1,1,4,Kırsal Mahalle,HATAY,HASSA,TUR031,HASSA,HASSA,...,6,249,9,10,0,0,0,0,95,81
2,Number of (heavily and more) damaged buildings...,2,2,9,Mahalle,GAZIANTEP,NIZIP,TUR027,NİZİP,NIZIP,...,4,153,5,0,0,0,0,0,9,2
3,Number of (heavily and more) damaged buildings...,3,3,10,Mahalle,GAZIANTEP,NIZIP,TUR027,NİZİP,NIZIP,...,48,1031,65,0,0,0,0,0,232,2
4,Number of (heavily and more) damaged buildings...,4,4,17,Kırsal Mahalle,KAHRAMANMARAS,PAZARCIK,TUR046,PAZARCIK,PAZARCIK,...,0,110,0,10,0,0,0,0,59,59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2946,Number of (heavily and more) damaged buildings...,1473,1473,3441,Kırsal Mahalle,HATAY,ANTAKYA,TUR031,ANTAKYA,ANTAKYA,...,0,21,0,3,0,0,0,0,55,50
2947,Number of (heavily and more) damaged buildings...,2674,2674,8124,Kırsal Mahalle,HATAY,ANTAKYA,TUR031,ANTAKYA,ANTAKYA,...,3,83,25,6,0,0,0,0,108,108
2948,Number of (heavily and more) damaged buildings...,2675,2675,8134,Mahalle,ADANA,CUKUROVA,TUR001,ÇUKUROVA,CUKUROVA,...,4,669,5,1,0,0,0,0,4,2
2949,Number of (heavily and more) damaged buildings...,2676,2676,8148,Mahalle,ADANA,CUKUROVA,TUR001,ÇUKUROVA,CUKUROVA,...,2,485,0,4,0,0,0,0,13,13


In [30]:
collapse = pd.merge(
    gdf, 
    collapse, 
    on=['adm1_en', 'adm1_tr',
       'adm1', 'adm2_tr', 'adm2_en', 'adm2', 'amd3_tr'], 
    how="outer", 
    indicator=True
)

In [31]:
collapse.groupby(['adm1_en', 'adm1_tr', 'adm1', 'adm2_tr', 'adm2_en', 'adm2', 'amd3_tr', "shape_leng", "shape_area"]).size().reset_index(name="count").sort_values("count", ascending=False)
collapse = collapse.sort_values(by="_merge", ascending=False).drop_duplicates(subset=['adm1_en', 'adm1_tr', 'adm1', 'adm2_tr', 'adm2_en', 'adm2', 'amd3_tr'], keep="first")
collapse

,TIP,adm1_en,adm1_tr,adm1,adm2_tr,adm2_en,adm2,amd3_tr,Shape_Leng,Shape_Area,...,no damage,not assessed,collapsed,out of scope,unidentified (unable to enter),registered/protected,moderate damage,total_dmg,total.bld,_merge
2704,Kırsal Mahalle,DIYARBAKIR,KAYAPINAR,TUR021,KAYAPINAR,KAYAPINAR,TUR021011,Yolboyu Pirinçlik Mh.,0.354830,0.004568,...,171.0,10.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,both
3568,Mahalle,GAZIANTEP,SAHINBEY,TUR027,ŞAHİNBEY,SAHINBEY,TUR027007,Binevler Mh.,0.058500,0.000098,...,318.0,16.0,0.0,0.0,0.0,0.0,0.0,58.0,8.0,both
3574,Kırsal Mahalle,GAZIANTEP,SAHINBEY,TUR027,ŞAHİNBEY,SAHINBEY,TUR027007,Burç Esentepe Mh.,0.243118,0.001577,...,714.0,3.0,0.0,0.0,0.0,0.0,0.0,165.0,4.0,both
3573,Mahalle,GAZIANTEP,SAHINBEY,TUR027,ŞAHİNBEY,SAHINBEY,TUR027007,Bozoklar Mh.,0.022178,0.000014,...,721.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,both
3572,Kırsal Mahalle,GAZIANTEP,SAHINBEY,TUR027,ŞAHİNBEY,SAHINBEY,TUR027007,Bozca Mh.,0.137183,0.001007,...,15.0,1.0,0.0,0.0,0.0,0.0,0.0,26.0,18.0,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2624,Kırsal Mahalle,DIYARBAKIR,HANI,TUR021,HANİ,HANI,TUR021009,Okurköy Mh.,0.230508,0.001766,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2621,Kırsal Mahalle,DIYARBAKIR,HANI,TUR021,HANİ,HANI,TUR021009,Kırım Mh.,0.343034,0.003082,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2620,Kırsal Mahalle,DIYARBAKIR,HANI,TUR021,HANİ,HANI,TUR021009,Kuyular Mh.,0.328090,0.004851,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2617,Kırsal Mahalle,DIYARBAKIR,HANI,TUR021,HANİ,HANI,TUR021009,Gürbüz Mh.,0.260753,0.002400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [17]:
collapse.columns

Index(['TIP', 'adm1_en', 'adm1_tr', 'adm1', 'adm2_tr', 'adm2_en', 'adm2',
       'amd3_tr', 'Shape_Leng', 'Shape_Area', 'geometry', 'Layer',
       'Feature_ID', 'qc_id', 'fid', 'tip', 'shape_leng', 'shape_area', 'adm3',
       'to be demolished urgently', 'heavy damage', 'light damage',
       'locked, not accessible', 'excluded from assessment', 'no damage',
       'not assessed', 'collapsed', 'out of scope',
       'unidentified (unable to enter)', 'registered/protected',
       ' moderate damage', 'total_dmg', '_merge'],
      dtype='object')

In [32]:
collapse['total.bld'] = collapse['collapsed'] + collapse['to be demolished urgently'] + collapse ['heavy damage']

In [34]:
collapse['total_dmg'].sum(skipna=True)

np.float64(205055.0)

In [35]:
collapse.to_excel("/Users/nihankaragul/Documents/Documents - Nihan’s MacBook Air/2024 Fall/Dissertation/Data/Processed/collapse_merge.xlsx", index=False)

In [9]:
collapse = collapse.drop(columns = ['_merge'])

In [10]:
collapse.to_file("/Users/nihankaragul/Documents/Documents - Nihan’s MacBook Air/2024 Fall/Dissertation/Data/collapse_shp/collapse.shp", driver="ESRI Shapefile")


/var/folders/xr/rxb54tks27nfmq0y661qwms80000gn/T/ipykernel_1822/831287596.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  collapse.to_file("/Users/nihankaragul/Documents/Documents - Nihan’s MacBook Air/2024 Fall/Dissertation/Data/collapse_shp/collapse.shp", driver="ESRI Shapefile")
